**Repository release note (2026-10-11):** Temporal VLE and assessment feature generation. Assessment-score inclusion uses submission/due-date proxies; actual score release timestamps are not observed. Interpret availability accordingly.

Saved execution outputs were removed from this public copy; source calculations have not been independently rerun. Configure `OULAD_PROJECT_DIR` or use the original Colab directory.


In [ ]:
import pandas as pd
import numpy as np
import os

BASE_PATH = os.environ.get("OULAD_PROJECT_DIR", "/content/drive/MyDrive/AI_Research_Paper_1")

DATA_PATH = os.path.join(BASE_PATH, "data")
RESULTS_PATH = os.path.join(BASE_PATH, "results")

print("Notebook 02 environment ready.")
print("studentVle exists:",
      os.path.exists(os.path.join(DATA_PATH, "studentVle.csv")))

In [ ]:
student_vle_sample = pd.read_csv(
    os.path.join(DATA_PATH, "studentVle.csv"),
    nrows=10
)

student_vle_sample

In [ ]:
print("Columns:")
print(student_vle_sample.columns.tolist())

print("\nData types:")
print(student_vle_sample.dtypes)

In [ ]:
vle_path = os.path.join(DATA_PATH, "studentVle.csv")

total_rows = 0
min_date = None
max_date = None
total_clicks = 0

for chunk in pd.read_csv(vle_path, chunksize=500_000):
    total_rows += len(chunk)

    chunk_min = chunk["date"].min()
    chunk_max = chunk["date"].max()

    min_date = chunk_min if min_date is None else min(min_date, chunk_min)
    max_date = chunk_max if max_date is None else max(max_date, chunk_max)

    total_clicks += chunk["sum_click"].sum()

print("Total studentVle rows:", f"{total_rows:,}")
print("Earliest interaction day:", min_date)
print("Latest interaction day:", max_date)
print("Total recorded clicks:", f"{total_clicks:,}")

In [ ]:
courses = pd.read_csv(os.path.join(DATA_PATH, "courses.csv"))
student_info = pd.read_csv(os.path.join(DATA_PATH, "studentInfo.csv"))
student_registration = pd.read_csv(
    os.path.join(DATA_PATH, "studentRegistration.csv")
)

keys = ["code_module", "code_presentation", "id_student"]

timing = (
    student_registration
    .merge(
        student_info[keys + ["final_result"]],
        on=keys,
        how="left"
    )
    .merge(
        courses,
        on=["code_module", "code_presentation"],
        how="left"
    )
)
issue_missing_registration = timing["date_registration"].isna()

issue_withdrawn_missing_date = (
    (timing["final_result"] == "Withdrawn") &
    timing["date_unregistration"].isna()
)

issue_fail_unregistration = (
    (timing["final_result"] == "Fail") &
    timing["date_unregistration"].notna()
)

issue_withdrawn_after_end = (
    (timing["final_result"] == "Withdrawn") &
    timing["date_unregistration"].notna() &
    (
        timing["date_unregistration"] >
        timing["module_presentation_length"]
    )
)

has_temporal_issue = (
    issue_missing_registration |
    issue_withdrawn_missing_date |
    issue_fail_unregistration |
    issue_withdrawn_after_end
)

clean_temporal = timing.loc[~has_temporal_issue].copy()

print("Clean temporal cohort:", len(clean_temporal))

In [ ]:
precourse_rows = 0
precourse_clicks = 0
precourse_students = set()

for chunk in pd.read_csv(vle_path, chunksize=500_000):

    pre = chunk[chunk["date"] < 0]

    precourse_rows += len(pre)
    precourse_clicks += pre["sum_click"].sum()

    precourse_students.update(
        zip(
            pre["code_module"],
            pre["code_presentation"],
            pre["id_student"]
        )
    )

print("Pre-course interaction rows:", f"{precourse_rows:,}")
print("Pre-course clicks:", f"{precourse_clicks:,}")
print("Students with pre-course activity:", f"{len(precourse_students):,}")

In [ ]:
keys = ["code_module", "code_presentation", "id_student"]

eligible_10 = clean_temporal.copy()

# 10% of each module's actual presentation length
eligible_10["cutoff_day"] = np.floor(
    eligible_10["module_presentation_length"] * 0.10
).astype(int)

# Student must already be registered
registered_by_cutoff = (
    eligible_10["date_registration"] <= eligible_10["cutoff_day"]
)

# Student must still be active at the cutoff
still_active = (
    eligible_10["date_unregistration"].isna() |
    (
        eligible_10["date_unregistration"] >
        eligible_10["cutoff_day"]
    )
)

eligible_10 = eligible_10[
    registered_by_cutoff & still_active
].copy()

print("Eligible students at 10%:", len(eligible_10))

print("\nOutcome distributiibution:")
print(eligible_10["final_result"].value_counts())

In [ ]:
eligible_lookup = eligible_10[
    keys + ["cutoff_day", "date_registration"]
].copy()

inc_click_parts = []
inc_day_parts = []
inc_site_parts = []

pre_click_parts = []
pre_day_parts = []
pre_site_parts = []

future_rows_excluded = 0
preregistration_rows_excluded = 0

for chunk in pd.read_csv(vle_path, chunksize=500_000):

    # Keep only students eligible at the 10% cutoff
    merged = chunk.merge(
        eligible_lookup,
        on=keys,
        how="inner"
    )

    # Count rows that would violate temporal availability
    future_rows_excluded += (
        merged["date"] > merged["cutoff_day"]
    ).sum()

    preregistration_rows_excluded += (
        merged["date"] < merged["date_registration"]
    ).sum()

    # -------------------------
    # PRE-COURSE ACTIVITY
    # -------------------------
    pre = merged[
        (merged["date"] < 0) &
        (merged["date"] >= merged["date_registration"])
    ].copy()

    if len(pre) > 0:

        pre_click_parts.append(
            pre.groupby(keys, as_index=False)["sum_click"].sum()
        )

        pre_day_parts.append(
            pre[keys + ["date"]].drop_duplicates()
        )

        pre_site_parts.append(
            pre[keys + ["id_site"]].drop_duplicates()
        )

    # -------------------------
    # IN-COURSE ACTIVITY
    # -------------------------
    inc = merged[
        (merged["date"] >= 0) &
        (merged["date"] >= merged["date_registration"]) &
        (merged["date"] <= merged["cutoff_day"])
    ].copy()

    if len(inc) > 0:

        inc_click_parts.append(
            inc.groupby(keys, as_index=False)["sum_click"].sum()
        )

        inc_day_parts.append(
            inc[keys + ["date"]].drop_duplicates()
        )

        inc_site_parts.append(
            inc[keys + ["id_site"]].drop_duplicates()
        )

print("Chunk processing completed.")
print("Future interaction rows excluded:",
      f"{future_rows_excluded:,}")
print("Pre-registration interaction rows excluded:",
      f"{preregistration_rows_excluded:,}")

In [ ]:
# ----- IN-COURSE CLICKS -----

inc_clicks = pd.concat(
    inc_click_parts,
    ignore_index=True
)

inc_clicks = (
    inc_clicks
    .groupby(keys, as_index=False)["sum_click"]
    .sum()
    .rename(columns={"sum_click": "vle_clicks_10"})
)


# ----- IN-COURSE ACTIVE DAYS -----

inc_days = pd.concat(
    inc_day_parts,
    ignore_index=True
).drop_duplicates()

inc_days = (
    inc_days
    .groupby(keys)
    .size()
    .reset_index(name="vle_active_days_10")
)


# ----- IN-COURSE UNIQUE SITES -----

inc_sites = pd.concat(
    inc_site_parts,
    ignore_index=True
).drop_duplicates()

inc_sites = (
    inc_sites
    .groupby(keys)
    .size()
    .reset_index(name="vle_unique_sites_10")
)


# ----- PRE-COURSE CLICKS -----

pre_clicks = pd.concat(
    pre_click_parts,
    ignore_index=True
)

pre_clicks = (
    pre_clicks
    .groupby(keys, as_index=False)["sum_click"]
    .sum()
    .rename(columns={"sum_click": "precourse_clicks"})
)


# ----- PRE-COURSE ACTIVE DAYS -----

pre_days = pd.concat(
    pre_day_parts,
    ignore_index=True
).drop_duplicates()

pre_days = (
    pre_days
    .groupby(keys)
    .size()
    .reset_index(name="precourse_active_days")
)


# ----- PRE-COURSE UNIQUE SITES -----

pre_sites = pd.concat(
    pre_site_parts,
    ignore_index=True
).drop_duplicates()

pre_sites = (
    pre_sites
    .groupby(keys)
    .size()
    .reset_index(name="precourse_unique_sites")
)

print("Student-level VLE features created.")

In [ ]:
vle_features_10 = eligible_10[
    keys + ["cutoff_day", "final_result"]
].copy()

for feature_table in [
    inc_clicks,
    inc_days,
    inc_sites,
    pre_clicks,
    pre_days,
    pre_sites
]:
    vle_features_10 = vle_features_10.merge(
        feature_table,
        on=keys,
        how="left"
    )

feature_columns = [
    "vle_clicks_10",
    "vle_active_days_10",
    "vle_unique_sites_10",
    "precourse_clicks",
    "precourse_active_days",
    "precourse_unique_sites"
]

vle_features_10[feature_columns] = (
    vle_features_10[feature_columns]
    .fillna(0)
    .astype(int)
)

print("Rows:", len(vle_features_10))
vle_features_10.head()

In [ ]:
print("Duplicate student records:",
      vle_features_10.duplicated(subset=keys).sum())

print("\nMissing feature values:")
print(vle_features_10[feature_columns].isna().sum())

print("\nFeature summary:")
print(
    vle_features_10[feature_columns]
    .describe()
    .round(2)
)

In [ ]:
print("Future interaction rows excluded:",
      f"{future_rows_excluded:,}")

print("Pre-registration interaction rows excluded:",
      f"{preregistration_rows_excluded:,}")

In [ ]:
vle_features_10[feature_columns].describe().T.round(2)

In [ ]:
zero_activity = pd.DataFrame({
    "Feature": feature_columns,
    "Students_with_zero": [
        (vle_features_10[col] == 0).sum()
        for col in feature_columns
    ],
    "Percent_zero": [
        round((vle_features_10[col] == 0).mean() * 100, 2)
        for col in feature_columns
    ]
})

zero_activity

In [ ]:
vle_features_10[feature_columns].describe().T.round(2)

In [ ]:
print("Future interaction rows excluded:",
      f"{future_rows_excluded:,}")

print("Pre-registration interaction rows excluded:",
      f"{preregistration_rows_excluded:,}")

In [ ]:
PROCESSED_PATH = os.path.join(BASE_PATH, "processed")

os.makedirs(PROCESSED_PATH, exist_ok=True)

vle_features_10.to_csv(
    os.path.join(PROCESSED_PATH, "vle_features_10.csv"),
    index=False
)

print("10% leakage-free VLE features saved successfully.")

In [ ]:
vle_info = pd.read_csv(
    os.path.join(DATA_PATH, "vle.csv")
)

print("Number of VLE activity types:",
      vle_info["activity_type"].nunique())

print("\nActivity types:")
print(
    vle_info["activity_type"]
    .value_counts()
)

In [ ]:
from collections import defaultdict

vle_map = vle_info[
    [
        "code_module",
        "code_presentation",
        "id_site",
        "activity_type"
    ]
].drop_duplicates()

activity_clicks = defaultdict(int)
activity_rows = defaultdict(int)
activity_students = defaultdict(set)

for chunk in pd.read_csv(vle_path, chunksize=500_000):

    # Keep only students eligible at 10%
    merged = chunk.merge(
        eligible_lookup,
        on=keys,
        how="inner"
    )

    # Strict leakage-free 10% window
    merged = merged[
        (merged["date"] >= 0) &
        (merged["date"] >= merged["date_registration"]) &
        (merged["date"] <= merged["cutoff_day"])
    ]

    # Add the VLE activity type
    merged = merged.merge(
        vle_map,
        on=["code_module", "code_presentation", "id_site"],
        how="left"
    )

    for activity, group in merged.groupby("activity_type"):

        activity_clicks[activity] += int(
            group["sum_click"].sum()
        )

        activity_rows[activity] += len(group)

        activity_students[activity].update(
            zip(
                group["code_module"],
                group["code_presentation"],
                group["id_student"]
            )
        )

print("10% activity-type audit completed.")

In [ ]:
activity_summary_10 = pd.DataFrame({
    "activity_type": list(activity_clicks.keys()),
    "total_clicks": [
        activity_clicks[a]
        for a in activity_clicks
    ],
    "interaction_rows": [
        activity_rows[a]
        for a in activity_clicks
    ],
    "students_using": [
        len(activity_students[a])
        for a in activity_clicks
    ]
})

activity_summary_10["student_coverage_percent"] = (
    activity_summary_10["students_using"]
    / len(eligible_10) * 100
).round(2)

activity_summary_10 = (
    activity_summary_10
    .sort_values("total_clicks", ascending=False)
    .reset_index(drop=True)
)

activity_summary_10

In [ ]:
activity_summary_10["click_share_percent"] = (
    activity_summary_10["total_clicks"]
    / activity_summary_10["total_clicks"].sum()
    * 100
).round(2)

print(
    activity_summary_10[
        [
            "activity_type",
            "total_clicks",
            "click_share_percent",
            "students_using",
            "student_coverage_percent"
        ]
    ].to_string(index=False)
)

In [ ]:
presentation_activity = (
    vle_info[
        [
            "code_module",
            "code_presentation",
            "activity_type"
        ]
    ]
    .drop_duplicates()
)

activity_availability = (
    presentation_activity
    .groupby("activity_type")
    .size()
    .reset_index(name="presentations_available")
)

total_presentations = (
    courses[
        ["code_module", "code_presentation"]
    ]
    .drop_duplicates()
    .shape[0]
)
activity_availability["availability_percent"] = (
    activity_availability["presentations_available"]
    / total_presentations
    * 100
).round(2)

activity_availability = (
    activity_availability
    .sort_values(
        "presentations_available",
        ascending=False
    )
)

print("Total module presentations:", total_presentations)

print(
    activity_availability.to_string(index=False)
)

In [ ]:
activity_group_map = {
    "oucontent": "content",
    "subpage": "content",
    "resource": "content",
    "forumng": "forum",
    "homepage": "navigation"
}

print(activity_group_map)

In [ ]:
def get_eligible_students(clean_temporal, progress):
    df = clean_temporal.copy()

    suffix = int(progress * 100)

    df["cutoff_day"] = np.floor(
        df["module_presentation_length"] * progress
    ).astype(int)

    registered = (
        df["date_registration"] <= df["cutoff_day"]
    )

    still_active = (
        df["date_unregistration"].isna() |
        (df["date_unregistration"] > df["cutoff_day"])
    )

    eligible = df[
        registered & still_active
    ].copy()

    print(
        f"{suffix}% cutoff: "
        f"{len(eligible):,} eligible students"
    )
    return eligible

In [ ]:
test_10 = get_eligible_students(clean_temporal, 0.10)

In [ ]:
def build_vle_features(progress):

    suffix = int(progress * 100)

    # ----------------------------------
    # 1. Eligible students at this cutoff
    # ----------------------------------
    eligible = get_eligible_students(
        clean_temporal,
        progress
    )

    lookup = eligible[
        keys + ["cutoff_day", "date_registration"]
    ].copy()

    # Storage for chunk results
    inc_click_parts = []
    inc_day_parts = []
    inc_site_parts = []

    pre_click_parts = []
    pre_day_parts = []
    pre_site_parts = []

    group_click_parts = []

    future_rows_excluded = 0
    preregistration_rows_excluded = 0

    # ----------------------------------
    # 2. Process studentVle in chunks
    # ----------------------------------
    for chunk in pd.read_csv(
        vle_path,
        chunksize=500_000
    ):

        merged = chunk.merge(
            lookup,
            on=keys,
            how="inner"
        )

        future_rows_excluded += (
            merged["date"] > merged["cutoff_day"]
        ).sum()

        preregistration_rows_excluded += (
            merged["date"] < merged["date_registration"]
        ).sum()

        # ------------------------------
        # PRE-COURSE ACTIVITY
        # ------------------------------
        pre = merged[
            (merged["date"] < 0) &
            (
                merged["date"] >=
                merged["date_registration"]
            )
        ].copy()

        if len(pre) > 0:

            pre_click_parts.append(
                pre.groupby(
                    keys,
                    as_index=False
                )["sum_click"].sum()
            )

            pre_day_parts.append(
                pre[keys + ["date"]]
                .drop_duplicates()
            )

            pre_site_parts.append(
                pre[keys + ["id_site"]]
                .drop_duplicates()
            )

        # ------------------------------
        # IN-COURSE ACTIVITY
        # ------------------------------
        inc = merged[
            (merged["date"] >= 0) &
            (
                merged["date"] >=
                merged["date_registration"]
            ) &
            (
                merged["date"] <=
                merged["cutoff_day"]
            )
        ].copy()

        if len(inc) > 0:

            inc_click_parts.append(
                inc.groupby(
                    keys,
                    as_index=False
                )["sum_click"].sum()
            )

            inc_day_parts.append(
                inc[keys + ["date"]]
                .drop_duplicates()
            )

            inc_site_parts.append(
                inc[keys + ["id_site"]]
                .drop_duplicates()
            )

            # Add activity type
            inc = inc.merge(
                vle_map,
                on=[
                    "code_module",
                    "code_presentation",
                    "id_site"
                ],
                how="left"
            )

            # Map universal activities
            inc["activity_group"] = (
                inc["activity_type"]
                .map(activity_group_map)
            )

            grouped = inc[
                inc["activity_group"].notna()
            ]

            if len(grouped) > 0:

                group_click_parts.append(
                    grouped.groupby(
                        keys + ["activity_group"],
                        as_index=False
                    )["sum_click"].sum()
                )

    # ----------------------------------
    # 3. Aggregate general VLE features
    # ----------------------------------

    inc_clicks = (
        pd.concat(
            inc_click_parts,
            ignore_index=True
        )
        .groupby(
            keys,
            as_index=False
        )["sum_click"]
        .sum()
        .rename(
            columns={
                "sum_click":
                f"vle_clicks_{suffix}"
            }
        )
    )

    inc_days = (
        pd.concat(
            inc_day_parts,
            ignore_index=True
        )
        .drop_duplicates()
        .groupby(keys)
        .size()
        .reset_index(
            name=f"vle_active_days_{suffix}"
        )
    )

    inc_sites = (
        pd.concat(
            inc_site_parts,
            ignore_index=True
        )
        .drop_duplicates()
        .groupby(keys)
        .size()
        .reset_index(
            name=f"vle_unique_sites_{suffix}"
        )
    )

    # ----------------------------------
    # 4. Aggregate pre-course features
    # ----------------------------------

    pre_clicks = (
        pd.concat(
            pre_click_parts,
            ignore_index=True
        )
        .groupby(
            keys,
            as_index=False
        )["sum_click"]
        .sum()
        .rename(
            columns={
                "sum_click":
                "precourse_clicks"
            }
        )
    )

    pre_days = (
        pd.concat(
            pre_day_parts,
            ignore_index=True
        )
        .drop_duplicates()
        .groupby(keys)
        .size()
        .reset_index(
            name="precourse_active_days"
        )
    )

    pre_sites = (
        pd.concat(
            pre_site_parts,
            ignore_index=True
        )
        .drop_duplicates()
        .groupby(keys)
        .size()
        .reset_index(
            name="precourse_unique_sites"
        )
    )

    # ----------------------------------
    # 5. Activity-group features
    # ----------------------------------

    group_clicks = pd.concat(
        group_click_parts,
        ignore_index=True
    )

    group_clicks = (
        group_clicks
        .groupby(
            keys + ["activity_group"],
            as_index=False
        )["sum_click"]
        .sum()
    )

    group_clicks = (
        group_clicks
        .pivot_table(
            index=keys,
            columns="activity_group",
            values="sum_click",
            fill_value=0
        )
        .reset_index()
    )

    group_clicks.columns.name = None

    group_clicks = group_clicks.rename(
        columns={
            "content":
                f"content_clicks_{suffix}",
            "forum":
                f"forum_clicks_{suffix}",
            "navigation":
                f"navigation_clicks_{suffix}"
        }
    )

    # ----------------------------------
    # 6. Merge into one student table
    # ----------------------------------

    features = eligible[
        keys + [
            "cutoff_day",
            "final_result"
        ]
    ].copy()

    feature_tables = [
        inc_clicks,
        inc_days,
        inc_sites,
        pre_clicks,
        pre_days,
        pre_sites,
        group_clicks
    ]

    for table in feature_tables:

        features = features.merge(
            table,
            on=keys,
            how="left"
        )

    feature_cols = [
        col for col in features.columns
        if col not in (
            keys +
            ["cutoff_day", "final_result"]
        )
    ]

    features[feature_cols] = (
        features[feature_cols]
        .fillna(0)
    )

    print(
        f"\n{suffix}% VLE dataset created."
    )

    print(
        "Rows:",
        f"{len(features):,}"
    )

    print(
        "Future rows excluded:",
        f"{future_rows_excluded:,}"
    )

    print(
        "Pre-registration rows excluded:",
        f"{preregistration_rows_excluded:,}"
    )

    print(
        "Duplicates:",
        features.duplicated(
            subset=keys
        ).sum()
    )

    return features

In [ ]:
vle_features_20 = build_vle_features(0.20)

In [ ]:
feature_cols_20 = [
    col for col in vle_features_20.columns
    if col not in (
        keys + ["cutoff_day", "final_result"]
    )
]

vle_features_20[feature_cols_20].describe().T.round(2)

In [ ]:
print("Missing values:")
print(vle_features_20[feature_cols_20].isna().sum())

print(
    "\nStudents with zero in-course clicks:",
    (vle_features_20["vle_clicks_20"] == 0).sum()
)

print(
    "Percent with zero in-course clicks:",
    round(
        (vle_features_20["vle_clicks_20"] == 0).mean() * 100,
        2
    )
)

In [ ]:
audit_20 = {
    "Progress": "20%",
    "Eligible_Students": len(vle_features_20),
    "Future_Rows_Excluded": 6555908,
    "PreRegistration_Rows_Excluded": 0,
    "Duplicates": vle_features_20.duplicated(subset=keys).sum(),
    "Zero_VLE_Students": (vle_features_20["vle_clicks_20"] == 0).sum(),
    "Zero_VLE_Percent": round(
        (vle_features_20["vle_clicks_20"] == 0).mean() * 100,
        2
    )
}

audit_20

In [ ]:
vle_features_30 = build_vle_features(0.30)

In [ ]:
audit_30 = {
    "Progress": "30%",
    "Eligible_Students": len(vle_features_30),
    "Future_Rows_Excluded": 5618122,
    "PreRegistration_Rows_Excluded": 0,
    "Duplicates": vle_features_30.duplicated(subset=keys).sum(),
    "Zero_VLE_Students": (vle_features_30["vle_clicks_30"] == 0).sum(),
    "Zero_VLE_Percent": round(
        (vle_features_30["vle_clicks_30"] == 0).mean() * 100,
        2
    )
}

audit_30

In [ ]:
vle_features_40 = build_vle_features(0.40)

In [ ]:
audit_40 = {
    "Progress": "40%",
    "Eligible_Students": len(vle_features_40),
    "Future_Rows_Excluded": 4706849,
    "PreRegistration_Rows_Excluded": 0,
    "Duplicates": vle_features_40.duplicated(subset=keys).sum(),
    "Zero_VLE_Students": (vle_features_40["vle_clicks_40"] == 0).sum(),
    "Zero_VLE_Percent": round(
        (vle_features_40["vle_clicks_40"] == 0).mean() * 100,
        2
    )
}

audit_40

In [ ]:
audit_10 = {
    "Progress": "10%",
    "Eligible_Students": 27652,
    "Future_Rows_Excluded": 7944900,
    "PreRegistration_Rows_Excluded": 0,
    "Duplicates": 0,
    "Zero_VLE_Students": 1319,
    "Zero_VLE_Percent": 4.77
}

In [ ]:
vle_temporal_audit = pd.DataFrame([
    audit_10,
    audit_20,
    audit_30,
    audit_40
])

vle_temporal_audit

In [ ]:
vle_features_20.to_csv(
    os.path.join(PROCESSED_PATH, "vle_features_20.csv"),
    index=False
)

vle_features_30.to_csv(
    os.path.join(PROCESSED_PATH, "vle_features_30.csv"),
    index=False
)

vle_features_40.to_csv(
    os.path.join(PROCESSED_PATH, "vle_features_40.csv"),
    index=False
)

print("20%, 30%, and 40% VLE feature datasets saved.")

In [ ]:
vle_temporal_audit.to_csv(
    os.path.join(RESULTS_PATH, "vle_temporal_audit.csv"),
    index=False
)

print("VLE temporal audit saved.")

In [ ]:
activity_summary_10.to_csv(
    os.path.join(RESULTS_PATH, "vle_activity_usage_10.csv"),
    index=False
)

activity_availability.to_csv(
    os.path.join(RESULTS_PATH, "vle_activity_availability.csv"),
    index=False
)

print("VLE activity audits saved.")

In [ ]:
assessments = pd.read_csv(
    os.path.join(DATA_PATH, "assessments.csv")
)

student_assessment = pd.read_csv(
    os.path.join(DATA_PATH, "studentAssessment.csv")
)

assessment_data = student_assessment.merge(
    assessments,
    on="id_assessment",
    how="left"
)

print("Assessment records:", f"{len(assessment_data):,}")

assessment_data.head()

In [ ]:
print("Columns:")
print(assessment_data.columns.tolist())

print("\nMissing values:")
print(assessment_data.isna().sum())

In [ ]:
print("Assessment types:")
print(
    assessment_data["assessment_type"]
    .value_counts()
)

print("\nBanked assessments:")
print(
    assessment_data["is_banked"]
    .value_counts(dropna=False)
)

In [ ]:
print("Submission timing:")
print(
    assessment_data["date_submitted"]
    .describe()
)

print("\nScheduled assessment dates:")
print(
    assessment_data["date"]
    .describe()
)

In [ ]:
print(
    "Submissions before course start:",
    (assessment_data["date_submitted"] < 0).sum()
)

print(
    "Submissions with missing scheduled date:",
    assessment_data["date"].isna().sum()
)

print(
    "Exam records:",
    (assessment_data["assessment_type"] == "Exam").sum()
)

In [ ]:
print("Missing score:",
      assessment_data["score"].isna().sum())

print("Missing date_submitted:",
      assessment_data["date_submitted"].isna().sum())

print("\n--- Missing scheduled date by assessment type ---")
print(
    pd.crosstab(
        assessment_data["assessment_type"],
        assessment_data["date"].isna()
    )
)

print("\n--- Pre-course submissions by banked status ---")
print(
    pd.crosstab(
        assessment_data["is_banked"],
        assessment_data["date_submitted"] < 0
    )
)

print("\n--- Pre-course submissions by assessment type ---")
print(
    pd.crosstab(
        assessment_data["assessment_type"],
        assessment_data["date_submitted"] < 0
    )
)

In [ ]:
precourse_nonbanked = assessment_data[
    (assessment_data["date_submitted"] < 0) &
    (assessment_data["is_banked"] == 0)
]

print(
    "Pre-course NON-BANKED assessment records:",
    len(precourse_nonbanked)
)

precourse_nonbanked[
    [
        "id_student",
        "id_assessment",
        "assessment_type",
        "date_submitted",
        "score",
        "date",
        "weight"
    ]
].head(20)

In [ ]:
assessment_timing = assessment_data.merge(
    student_registration[
        [
            "code_module",
            "code_presentation",
            "id_student",
            "date_registration"
        ]
    ],
    on=[
        "code_module",
        "code_presentation",
        "id_student"
    ],
    how="left"
)

print(
    "Assessment submissions before registration:",
    (
        assessment_timing["date_submitted"] <
        assessment_timing["date_registration"]
    ).sum()
)

print(
    "\nPre-course non-banked submissions before registration:",
    (
        (assessment_timing["date_submitted"] < 0) &
        (assessment_timing["is_banked"] == 0) &
        (
            assessment_timing["date_submitted"] <
            assessment_timing["date_registration"]
        )
    ).sum()
)

In [ ]:
print("Missing score:",
      assessment_data["score"].isna().sum())

print("Missing date_submitted:",
      assessment_data["date_submitted"].isna().sum())

print("\nMissing scheduled date by assessment type:")
print(
    pd.crosstab(
        assessment_data["assessment_type"],
        assessment_data["date"].isna()
    )
)

print("\nPre-course submissions by banked status:")
print(
    pd.crosstab(
        assessment_data["is_banked"],
        assessment_data["date_submitted"] < 0
    )
)

In [ ]:
print("Missing scores by assessment type:")
print(
    assessment_data.loc[
        assessment_data["score"].isna(),
        "assessment_type"
    ].value_counts()
)

print("\nMissing scores by banked status:")
print(
    assessment_data.loc[
        assessment_data["score"].isna(),
        "is_banked"
    ].value_counts()
)

assessment_data[
    assessment_data["score"].isna()
][
    [
        "id_student",
        "id_assessment",
        "assessment_type",
        "date_submitted",
        "is_banked",
        "date",
        "weight"
    ]
].head(20)

In [ ]:
assessment_timing = assessment_data.merge(
    student_registration[
        [
            "code_module",
            "code_presentation",
            "id_student",
            "date_registration"
        ]
    ],
    on=[
        "code_module",
        "code_presentation",
        "id_student"
    ],
    how="left"
)

print(
    "Assessment submissions before registration:",
    (
        assessment_timing["date_submitted"] <
        assessment_timing["date_registration"]
    ).sum()
)

print(
    "Pre-course NON-BANKED submissions before registration:",
    (
        (assessment_timing["date_submitted"] < 0) &
        (assessment_timing["is_banked"] == 0) &
        (
            assessment_timing["date_submitted"] <
            assessment_timing["date_registration"]
        )
    ).sum()
)

In [ ]:
missing_score_audit = assessment_data[
    assessment_data["score"].isna()
].copy()

print("Total missing scores:", len(missing_score_audit))

print("\nMissing scores by assessment type:")
print(
    missing_score_audit["assessment_type"]
    .value_counts()
)

print("\nMissing scores by banked status:")
print(
    missing_score_audit["is_banked"]
    .value_counts(dropna=False)
)

print("\nMissing scores by submission timing:")
print(
    missing_score_audit["date_submitted"]
    .describe()
)

In [ ]:
missing_score_audit[
    [
        "id_student",
        "id_assessment",
        "assessment_type",
        "date_submitted",
        "is_banked",
        "date",
        "weight"
    ]
].head(20)

In [ ]:
def build_assessment_features(progress):

    suffix = int(progress * 100)

    # -----------------------------
    # 1. Eligible students
    # -----------------------------
    eligible = get_eligible_students(
        clean_temporal,
        progress
    )

    lookup = eligible[
        keys + ["cutoff_day", "date_registration"]
    ].copy()

    # Attach cutoff information
    data = assessment_data.merge(
        lookup,
        on=keys,
        how="inner"
    )

    # Audit exclusions
    banked_excluded = (
        data["is_banked"] == 1
    ).sum()

    exams_excluded = (
        data["assessment_type"] == "Exam"
    ).sum()

    # -----------------------------
    # 2. Current-course TMA/CMA only
    # -----------------------------
    current = data[
        (data["is_banked"] == 0) &
        (
            data["assessment_type"]
            .isin(["TMA", "CMA"])
        )
    ].copy()

    future_submissions = (
        current["date_submitted"] >
        current["cutoff_day"]
    ).sum()

    before_registration = (
        current["date_submitted"] <
        current["date_registration"]
    ).sum()

    # Only genuinely submitted by cutoff
    observed = current[
        (
            current["date_submitted"] >=
            current["date_registration"]
        ) &
        (
            current["date_submitted"] <=
            current["cutoff_day"]
        )
    ].copy()

    # -----------------------------
    # 3. Strict timing features
    # -----------------------------
    observed["late_flag"] = (
        observed["date_submitted"] >
        observed["date"]
    ).astype(int)

    observed["submission_delay"] = (
        observed["date_submitted"] -
        observed["date"]
    )

    timing_features = (
        observed
        .groupby(keys)
        .agg(
            assessments_submitted=(
                "id_assessment", "nunique"
            ),
            tma_submitted=(
                "assessment_type",
                lambda x: (x == "TMA").sum()
            ),
            cma_submitted=(
                "assessment_type",
                lambda x: (x == "CMA").sum()
            ),
            late_submissions=(
                "late_flag", "sum"
            ),
            mean_submission_delay=(
                "submission_delay", "mean"
            )
        )
        .reset_index()
    )

    # -----------------------------
    # 4. Conservative score rule
    # -----------------------------
    score_available = observed[
        observed["score"].notna() &
        observed["date"].notna() &
        (
            observed["date"] <=
            observed["cutoff_day"]
        )
    ].copy()

    # Scores excluded because the
    # scheduled assessment date is future
    scores_not_due_yet = (
        observed["score"].notna() &
        (
            observed["date"] >
            observed["cutoff_day"]
        )
    ).sum()

    score_available["weighted_component"] = (
        score_available["score"] *
        score_available["weight"]
    )

    score_features = (
        score_available
        .groupby(keys)
        .agg(
            scored_assessments=(
                "score", "count"
            ),
            mean_score_due=(
                "score", "mean"
            ),
            weighted_score_sum=(
                "weighted_component", "sum"
            ),
            weight_sum=(
                "weight", "sum"
            )
        )
        .reset_index()
    )

    score_features["weighted_mean_score_due"] = (
        score_features["weighted_score_sum"] /
        score_features["weight_sum"].replace(
            0, np.nan
        )
    )

    score_features = score_features.drop(
        columns=[
            "weighted_score_sum",
            "weight_sum"
        ]
    )

    # -----------------------------
    # 5. Merge onto all eligible students
    # -----------------------------
    features = eligible[
        keys + ["cutoff_day", "final_result"]
    ].copy()

    features = features.merge(
        timing_features,
        on=keys,
        how="left"
    )

    features = features.merge(
        score_features,
        on=keys,
        how="left"
    )

    # Counts can safely be zero
    count_columns = [
        "assessments_submitted",
        "tma_submitted",
        "cma_submitted",
        "late_submissions",
        "scored_assessments"
    ]

    features[count_columns] = (
        features[count_columns]
        .fillna(0)
        .astype(int)
    )

    # Helpful availability indicator
    features["has_score_due"] = (
        features["scored_assessments"] > 0
    ).astype(int)

    # Add cutoff suffix
    rename_map = {
        col: f"{col}_{suffix}"
        for col in features.columns
        if col not in (
            keys +
            ["cutoff_day", "final_result"]
        )
    }

    features = features.rename(
        columns=rename_map
    )

    print(f"\n{suffix}% assessment dataset created.")
    print("Rows:", f"{len(features):,}")
    print("Banked records excluded:",
          f"{banked_excluded:,}")
    print("Exam records excluded:",
          f"{exams_excluded:,}")
    print("Future submissions excluded:",
          f"{future_submissions:,}")
    print("Before-registration records excluded:",
          f"{before_registration:,}")
    print("Scores not yet due excluded:",
          f"{scores_not_due_yet:,}")
    print(
        "Duplicates:",
        features.duplicated(
            subset=keys
        ).sum()
    )

    return features

In [ ]:
assessment_features_10 = build_assessment_features(0.10)

In [ ]:
assessment_features_10.head()

In [ ]:
print(
    assessment_features_10.isna().sum()
)

print(
    "\nDuplicate records:",
    assessment_features_10
    .duplicated(subset=keys)
    .sum()
)

In [ ]:
print(
    "Students with no submitted assessment:",
    (assessment_features_10["assessments_submitted_10"] == 0).sum()
)

print(
    "Students with no available score:",
    (assessment_features_10["scored_assessments_10"] == 0).sum()
)

weighted_only_missing = assessment_features_10[
    assessment_features_10["mean_score_due_10"].notna() &
    assessment_features_10["weighted_mean_score_due_10"].isna()
]

print(
    "Students with mean score available but weighted mean missing:",
    len(weighted_only_missing)
)

In [ ]:
zero_weight = assessment_data[
    (assessment_data["is_banked"] == 0) &
    (assessment_data["assessment_type"].isin(["TMA", "CMA"])) &
    (assessment_data["weight"] == 0)
]

print("Non-banked TMA/CMA records with weight = 0:",
      len(zero_weight))

print("\nAssessment IDs with zero weight:")
print(
    zero_weight[
        ["code_module", "code_presentation",
         "id_assessment", "assessment_type", "weight"]
    ]
    .drop_duplicates()
    .head(30)
)

In [ ]:
assessment_features_10 = assessment_features_10.drop(
    columns=["weighted_mean_score_due_10"]
)

print("10% weighted score feature removed.")

print("\nRemaining columns:")
print(assessment_features_10.columns.tolist())

In [ ]:
assessment_features_10.to_csv(
    os.path.join(
        PROCESSED_PATH,
        "assessment_features_10.csv"
    ),
    index=False
)

print("10% assessment features saved.")

In [ ]:
assessment_features_20 = build_assessment_features(0.20)

In [ ]:
assessment_features_20 = assessment_features_20.drop(
    columns=["weighted_mean_score_due_20"]
)

In [ ]:
assessment_features_30 = build_assessment_features(0.30)

In [ ]:
assessment_features_30 = assessment_features_30.drop(
    columns=["weighted_mean_score_due_30"]
)

In [ ]:
assessment_features_40 = build_assessment_features(0.40)

In [ ]:
assessment_features_40 = assessment_features_40.drop(
    columns=["weighted_mean_score_due_40"]
)

In [ ]:
assessment_features_20.to_csv(
    os.path.join(PROCESSED_PATH, "assessment_features_20.csv"),
    index=False
)

assessment_features_30.to_csv(
    os.path.join(PROCESSED_PATH, "assessment_features_30.csv"),
    index=False
)

assessment_features_40.to_csv(
    os.path.join(PROCESSED_PATH, "assessment_features_40.csv"),
    index=False
)

print("20%, 30%, and 40% assessment datasets saved.")

In [ ]:
assessment_temporal_audit = pd.DataFrame({
    "Progress": ["10%", "20%", "30%", "40%"],
    "Eligible_Students": [
        27652,
        26595,
        25783,
        25119
    ],
    "Banked_Records_Excluded": [
        1801,
        1774,
        1743,
        1724
    ],
    "Exam_Records_Excluded": [
        4956,
        4956,
        4957,
        4959
    ],
    "Future_Submissions_Excluded": [
        145445,
        125656,
        106605,
        89460
    ],
    "Before_Registration_Excluded": [
        0, 0, 0, 0
    ],
    "Scores_Not_Yet_Due_Excluded": [
        3405,
        6654,
        6477,
        9008
    ],
    "Duplicates": [
        0, 0, 0, 0
    ]
})

assessment_temporal_audit

In [ ]:
assessment_temporal_audit.to_csv(
    os.path.join(
        RESULTS_PATH,
        "assessment_temporal_audit.csv"
    ),
    index=False
)

print("Assessment temporal audit saved.")

In [ ]:
def check_feature_alignment(vle_df, assessment_df, progress):

    vle_keys = set(
        map(
            tuple,
            vle_df[keys].to_numpy()
        )
    )

    assessment_keys = set(
        map(
            tuple,
            assessment_df[keys].to_numpy()
        )
    )

    print(f"{progress}% cutoff")
    print(
        "VLE students:",
        f"{len(vle_keys):,}"
    )
    print(
        "Assessment students:",
        f"{len(assessment_keys):,}"
    )
    print(
        "Exact match:",
        vle_keys == assessment_keys
    )
    print(
        "Only in VLE:",
        len(vle_keys - assessment_keys)
    )
    print(
        "Only in assessment:",
        len(assessment_keys - vle_keys)
    )
    print("-" * 40)

In [ ]:
check_feature_alignment(
    vle_features_10,
    assessment_features_10,
    10
)

check_feature_alignment(
    vle_features_20,
    assessment_features_20,
    20
)

check_feature_alignment(
    vle_features_30,
    assessment_features_30,
    30
)

check_feature_alignment(
    vle_features_40,
    assessment_features_40,
    40
)